# KG1 v149 - TONG SAFE max_seq=3072 (alvo 0.85-0.88)

## Por que v149 vs v148?

v147 estava treinando com config ERRADA que ia regredir vs 0.86 atual:
- max_seq=2560 -> truncava 50%+ dos samples (perdia o `\boxed{}` final!)
- lr=5e-5 -> 4x MENOR que Tong validated (2e-4) -> undertfit
- Dataset misturado com cryptarithm/eq/synth nao validados

## v149 = Tong recipe SAFE (max_seq=3072 cabe sem OOM):

| Parametro | v147 (errado) | **v149 (Tong safe)** |
|---|---|---|
| max_seq | 2560 (trunca) | **3072** (cabe em 80GB sem OOM) |
| learning_rate | 5e-5 | **2e-4** (Tong validated) |
| epochs | 2 | **1** (Tong validated) |
| Dataset | tong + 5388 nao validados | **APENAS Tong puro** |
| Optimizer | adamw_torch_fused | adamw_torch_fused (igual) |
| Liger-Kernel | tentou v148 (falhou) | **NAO** (Nemotron-H custom CE incompativel) |

## VRAM v149 (sem Liger - max_seq=3072):

```
Modelo BF16:                60 GB
LoRA + grads:                0.5 GB
Activations seq=3072:        4 GB (sem Liger, max_seq menor)
adamw_torch_fused states:    3.5 GB
CUDA workspace:              2 GB
TOTAL:                      ~70 GB / 80 GB
Margem:                      ~10 GB SEGURA
```

## Score esperado v149 (max_seq=3072):

- **Pessimista (35%)**: 0.78-0.83 (max_seq menor que Tong original 8192)
- **Neutro (40%)**: 0.83-0.86 (empate com 0.86 atual)
- **Otimista (20%)**: 0.86-0.88 (LR correto + dataset puro compensa max_seq)
- **Melhor caso (5%)**: > 0.88

## Setup

1. **Runtime -> A100 ALTA RAM** (80 GB)
2. **Colab Secrets**: `HF_KEY` ja configurado
3. **Run all** (1 cell unica)

Tempo: ~2.5-3.5h A100 (com adamw_fused, max_seq=3072)


In [ ]:
# CELL UNICA v149 - Tong recipe SAFE max_seq=3072
import os, sys, time, math, gc, glob, csv, json, urllib.request, statistics, subprocess

# === ENV configs ===
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

print('=' * 70)
print('KG1 v149 - TONG SAFE (lr=2e-4, max_seq=3072, 1 epoch)')
print('=' * 70)

# ============================================================
# STEP 1: Token + GPU + secrets
# ============================================================
print('\n[1/9] Setup secrets + GPU check')
try:
    from google.colab import userdata
    hf_token = None
    for k in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(k)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                print(f'  HF token via {k} ({len(v)} chars)')
                break
        except: pass
    if not hf_token:
        raise RuntimeError('HF_KEY missing - configure Colab Secret')

    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except: pass
except ImportError:
    raise RuntimeError('Run no Colab')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required')

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB')

if vram_gb < 70:
    raise RuntimeError(f'VRAM={vram_gb:.1f}GB insuficiente. Precisa A100 ALTA RAM (80GB)')

# ============================================================
# STEP 2: Install dependencies (com Liger-Kernel)
# ============================================================
print('\n[2/9] Install dependencies (com Liger-Kernel)')
deps = [
    'transformers>=4.48.0', 'peft>=0.14.0', 'trl>=0.14.0', 'datasets>=3.0.0',
    'accelerate>=1.3.0', 'bitsandbytes>=0.45.0', 'huggingface_hub>=0.27.0',
    'torchao>=0.16.0', 'einops', 'packaging', 'ninja', 'triton',
    'liger-kernel>=0.5.0',  # NOVO: fused CE para economizar ~5 GB VRAM
]
for pkg in deps:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                      capture_output=True, text=True, timeout=300)
    print(f'  {"[OK]" if r.returncode == 0 else "[FAIL]"} {pkg}')

# Force reload modules
for m in ['transformers', 'peft', 'trl', 'torchao', 'bitsandbytes', 'liger_kernel']:
    if m in sys.modules:
        del sys.modules[m]

# ============================================================
# STEP 3: Install mamba-ssm v2.3.1 + causal-conv1d v1.6.1
# ============================================================
print('\n[3/9] Install mamba-ssm 2.3.1 + causal-conv1d 1.6.1')
import torch
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'
print(f'  Detected: {py_ver} | torch{torch_short} | abi{abi_str}')

attempts = [('cu12', torch_short, abi_str)]
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        if (('cu12', t, abi)) not in attempts:
            attempts.append(('cu12', t, abi))

success = False
for cu, t, abi in attempts:
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    ok_cc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', cc_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    ok_mb = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', mb_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    if ok_cc and ok_mb:
        print(f'  [OK] mamba_ssm 2.3.1 + causal_conv1d 1.6.1 ({cu} torch{t} abi{abi})')
        success = True
        break

if not success:
    raise RuntimeError('mamba-ssm install failed')

for m in ['mamba_ssm', 'causal_conv1d', 'selective_scan_cuda', 'causal_conv1d_cuda']:
    if m in sys.modules:
        del sys.modules[m]

import mamba_ssm, causal_conv1d
print(f'  mamba_ssm: {mamba_ssm.__version__}')

# ============================================================
# STEP 4: Download Tong corpus PURO (sem misturar com nao-validados)
# ============================================================
print('\n[4/9] Download Tong corpus puro')
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

# APENAS Tong corpus (validado LB 0.84 no V70)
TONG_URL = 'https://huggingface.co/datasets/felipesp1983/nemotron-cot-tong-dgxchen/resolve/main/less_cot.csv'
out = os.path.join(DATA_DIR, 'tong.csv')
urllib.request.urlretrieve(TONG_URL, out)
sz = os.path.getsize(out)
print(f'  [OK] tong.csv: {sz/1e6:.2f} MB')

# ============================================================
# STEP 5: Format Tong dataset (sem mixing)
# ============================================================
print('\n[5/9] Format Tong dataset (puro, sem mixing)')
from datasets import Dataset

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
all_data = []

with open(os.path.join(DATA_DIR, 'tong.csv'), encoding='utf-8') as f:
    rd = csv.DictReader(f)
    for row in rd:
        p = (row.get('prompt') or '').strip()
        c = (row.get('generated_cot') or '').strip()
        if p and c:
            all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c})

ds = Dataset.from_list(all_data)
print(f'  TOTAL: {len(ds)} samples (Tong puro - igual V70 que deu LB 0.84)')

# ============================================================
# STEP 6: Load model BF16 + Liger-Kernel (fused CE)
# ============================================================
print(f'\n[6/9] Load Nemotron-30B em BF16 + Liger-Kernel')
from huggingface_hub import HfApi
whoami = HfApi(token=hf_token).whoami()
print(f'  HF user: {whoami["name"]}')

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

print('  Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'  Loading model 30B BF16...')
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    token=hf_token,
    attn_implementation='eager',
)
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': True})
model.config.use_cache = False
print(f'  [OK] Model loaded in {time.time()-t0:.1f}s')
print(f'  VRAM after model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# === Apply Liger-Kernel fused CE (CRITICO para economizar VRAM em max_seq=4096) ===
print('\n  Applying Liger-Kernel fused linear+CE (economiza ~5 GB)...')
try:
    from liger_kernel.transformers import _apply_liger_kernel_to_instance
    _apply_liger_kernel_to_instance(model=model, fused_linear_cross_entropy=True)
    print('  [OK] Liger-Kernel fused CE applied')
except Exception as e:
    print(f'  [WARN] Liger-Kernel falhou: {e}')
    print('  [WARN] Continuando sem Liger - VRAM pode ficar apertada')

# Apply LoRA com Tong targets
TARGET_REGEX = r'.*\.(in_proj|out_proj|q_proj|k_proj|v_proj|o_proj|up_proj|down_proj|gate_proj|lm_head)$'
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=32, target_modules=TARGET_REGEX,
    lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'  [OK] LoRA r=32 a=32: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.3f}%)')
print(f'  VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB / {vram_gb:.1f} GB')

# ============================================================
# STEP 7: Tokenize com max_length=4096 (Tong = 8192, mas com Liger 4096 cobre ~90% samples)
# ============================================================
print('\n[7/9] Tokenize (max_length=4096 - Liger permite!)')
MAX_LENGTH = 3072

def fmt(ex):
    messages = [
        {'role': 'user', 'content': ex['prompt']},
        {'role': 'assistant', 'content': ex['completion']},
    ]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=True)
    prompt_text = tokenizer.apply_chat_template([messages[0]], tokenize=False, add_generation_prompt=True, enable_thinking=True)
    full_ids = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH, return_tensors=None, add_special_tokens=False)['input_ids']
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=MAX_LENGTH, return_tensors=None, add_special_tokens=False)['input_ids']
    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100
    return {'input_ids': full_ids, 'attention_mask': [1] * len(full_ids), 'labels': labels}

t0 = time.time()
tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
print(f'  [OK] Tokenized {len(tokenized)} samples in {time.time()-t0:.1f}s')

seq_lens = [len(tokenized[i]['input_ids']) for i in range(min(500, len(tokenized)))]
truncated = sum(1 for l in seq_lens if l == MAX_LENGTH)
print(f'  Sample lengths: min={min(seq_lens)} median={statistics.median(seq_lens):.0f} max={max(seq_lens)}')
print(f'  Truncated at {MAX_LENGTH}: {truncated}/{len(seq_lens)} samples ({100*truncated/len(seq_lens):.1f}%)')

# ============================================================
# STEP 8: TRAIN com Tong recipe correto
# ============================================================
print('\n[8/9] TRAIN (TONG RECIPE: lr=2e-4, 1 epoch, cosine warmup 5%)')
gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR = '/content/output_v149'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PAD_TO_MULTIPLE_OF = 8
PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

def custom_collator(features):
    max_len = max(len(f['input_ids']) for f in features)
    max_len = ((max_len + PAD_TO_MULTIPLE_OF - 1) // PAD_TO_MULTIPLE_OF) * PAD_TO_MULTIPLE_OF
    input_ids_batch, attention_mask_batch, labels_batch = [], [], []
    for f in features:
        pad_len = max_len - len(f['input_ids'])
        input_ids_batch.append(f['input_ids'] + [PAD_TOKEN_ID] * pad_len)
        attention_mask_batch.append(f['attention_mask'] + [0] * pad_len)
        labels_batch.append(f['labels'] + [-100] * pad_len)
    return {
        'input_ids': torch.tensor(input_ids_batch, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask_batch, dtype=torch.long),
        'labels': torch.tensor(labels_batch, dtype=torch.long),
    }

from transformers import Trainer, TrainingArguments, TrainerCallback

class LossExplosionCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            l = logs['loss']
            if l > 15.0 or (l != l):
                print(f'\n[WARN] Loss explosion at step {state.global_step}: {l}')
                control.should_training_stop = True

N_TRAIN = len(tokenized)
EFFECTIVE_BATCH = 32  # Tong recipe
N_STEPS = math.ceil(N_TRAIN / EFFECTIVE_BATCH) * 1  # 1 EPOCH (Tong validated)
WARMUP_STEPS = int(N_STEPS * 0.05)  # 5% warmup

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,                          # TONG: 1 epoch
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,              # effective batch = 32
    learning_rate=2e-4,                          # TONG VALIDATED (NAO 5e-5!)
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP_STEPS,
    adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
    weight_decay=0.0,                            # TONG: 0.0 (NAO 0.01)
    max_grad_norm=1.0,
    logging_steps=10,
    save_steps=max(50, N_STEPS // 4),
    save_total_limit=4,
    bf16=True,
    optim='adamw_torch_fused',                   # proven em v147
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
    remove_unused_columns=False, report_to='none',
    dataloader_num_workers=0,
    seed=42,
)

trainer = Trainer(
    model=model, args=train_args,
    train_dataset=tokenized, data_collator=custom_collator,
    callbacks=[LossExplosionCallback()],
)
print(f'  Steps: {N_STEPS} | Warmup: {WARMUP_STEPS} | LR: 2e-4 (TONG)')
print(f'  Time estimate: ~3-4h A100 (1 epoch)')
print()

t0 = time.time()
trainer.train()
train_time_min = (time.time() - t0) / 60
print(f'\n[OK] Training complete in {train_time_min:.1f} min')

# ============================================================
# STEP 9: Save + Upload + Backup
# ============================================================
print('\n[9/9] Save + Upload HF + GDrive backup')
ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'  [OK] Saved at {ADAPTER_DIR}')

# Upload HF
try:
    from huggingface_hub import HfApi, create_repo
    DEST_REPO = 'felipesp1983/kg1-v149-tong-safe'
    try:
        from google.colab import userdata
        for k in ['HF_KEY', 'HF_TOKEN']:
            try:
                v = userdata.get(k)
                if v:
                    hf_token = v
                    break
            except: pass
    except: pass
    api = HfApi(token=hf_token)
    create_repo(DEST_REPO, private=False, exist_ok=True, token=hf_token)
    api.upload_folder(folder_path=ADAPTER_DIR, repo_id=DEST_REPO, repo_type='model',
                      commit_message=f'v149 Tong safe lr=2e-4 max_seq=3072 1ep {train_time_min:.1f}min')
    print(f'  [OK] HF: https://huggingface.co/{DEST_REPO}')
except Exception as e:
    print(f'  [WARN] HF upload: {e}')

# GDrive backup
try:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    GD = '/content/drive/My Drive/KG1_v149_adapter'
    if os.path.exists(GD):
        shutil.rmtree(GD)
    shutil.copytree(ADAPTER_DIR, GD)
    print(f'  [OK] GDrive: {GD}')
except Exception as e:
    print(f'  [WARN] GDrive: {e}')

print('\n' + '=' * 70)
print('SUCCESS - v149 Tong safe training complete!')
print(f'Adapter: https://huggingface.co/felipesp1983/kg1-v149-tong-safe')
print(f'Target Kaggle LB: 0.86-0.88 (vs atual 0.86)')
print('=' * 70)
